In [1]:
import json
import numpy as np
import os

from Historia.shared.design_utils import read_labels
from Historia.shared.design_utils import lhd

In [2]:
folder_experiment_name = "HCM/2/scenarios/37"
basefolder             = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
datafolder             = f"{basefolder}/data"

# fields = ['ToRORd',
# 		  'ToRORd_land',
# 		  'EP',
# 		  'mechanics',
# 		  'circadapt'
# 		  ]

# fields = ['mechanics']
fields = ['EP']

# folder_experiment_name = "rodero_healthy/h11/scenarios/9"
# basefolder             = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
# datafolder             = f"{basefolder}/data"

# fields = ['EP',
# 		  'mechanics',
# 		  'circadapt'
# 		  ]

# folder_experiment_name = "rodero_healthy/h11/scenarios/calibration_game/level3/"
# basefolder             = f"/media/croderog/SeagateExpansionDrive/{folder_experiment_name}"
# datafolder             = f"{basefolder}/data"

# fields = ['mechanics',
# 		  'circadapt'
# 		  ]

Read parameter ranges and labels and sample using LHD:

In [4]:
N = 1000 # To play safe


for f in fields:
    I_ = np.loadtxt(f"{datafolder}/I_{f}.txt",dtype=float)

    xlabels_ = read_labels(f"{datafolder}/xlabels_{f}.txt")

    # check if dimensions match
    if len(xlabels_)!=I_.shape[0] and len(xlabels_) > 1: 
        print(f"Read {datafolder}/I_{f}.txt")
        print("xlabels: ")
        print(xlabels_)
        print("Intervals: ")
        print(I_)
        raise Exception(f"xlabels and I sizes for field {f} do not match, xlabels is {len(xlabels_)} and intervals are {I_.shape[0]}")

    X_ = lhd(np.atleast_2d(I_),N)
    np.savetxt(f"{datafolder}/X_{f}.txt",X_,fmt="%g")

Convert to json files for later:

In [3]:
def X_to_json_modified(labels_fields,
			  datafolder,
			  outputfolder,
			  default_json):

	print('generating json file...')

	os.system('mkdir '+outputfolder)

	
	N = None
	while N is None:
		for lab in labels_fields:
			print(lab)
			X_tmp = np.loadtxt(datafolder+'/X_'+lab+'.txt')
			labels = read_labels(datafolder+'/xlabels_'+lab+'.txt')	
			# print(lab)
			# print(X_tmp.shape)
			if len(X_tmp.shape)>1 or len(labels) == 1:
				N = X_tmp.shape[0]
			elif (len(X_tmp.shape) == 1) and len(labels)>1:
				N = 1

	# generate this dictionary to avoid reading X_*.txt at every iteration
	dct_datasets = {}
	for k,lab in enumerate(labels_fields):
		print(lab)
		dct_datasets[lab] = {}

		labels = read_labels(datafolder+'/xlabels_'+lab+'.txt')	
		dct_datasets[lab]["labels"] = labels

		X = np.loadtxt(datafolder+'/X_'+lab+'.txt')
		dct_datasets[lab]["X"] = X

	for i in range(N):
		# if you want to combine the new parameters
		# with the default json file, then the dictionary
		# is initialised to the default json you give.
		# Otherwise it's empty

		f_input = open(default_json,"r")
		param_dictionary = json.load(f_input)
		f_input.close()

		for k,lab in enumerate(labels_fields):

			labels = dct_datasets[lab]["labels"]
			X = dct_datasets[lab]["X"]

			if default_json is not None:
				subdict = param_dictionary[lab]
			else:
				subdict = {}
			
			if (len(X.shape) == 1) and N==1:
				X = X.reshape(1,X.shape[0])
			elif (len(X.shape) == 1) and N>1:
				X = X.reshape(N,1)

			if (len(labels)!=X.shape[1]):
				raise ValueError('xlabels_'+lab+'.txt'+' and X_'+lab+'.txt do not match')	
			if (X.shape[0]!=N):
				raise ValueError('X_'+lab+'.txt'+' and X_'+labels_fields[0]+'.txt do not match')		
			for j in range(len(labels)):
				subdict[labels[j]] = X[i,j]	


			param_dictionary[lab] = subdict

		with open(outputfolder+'/'+str(i)+'.json', 'w') as f:
		    json.dump(param_dictionary, f, indent=4)


In [7]:
X_to_json_modified(labels_fields = fields,
                          datafolder    = datafolder,
                          outputfolder  = f"{basefolder}/json_files",
                          default_json  = f"{basefolder}/json_files/default.json")

generating json file...
EP
EP


mkdir: cannot create directory ‘/media/croderog/SeagateExpansionDrive/HCM/2/scenarios/37/json_files’: File exists
